# Predictive Coding in JAX 🤖🧠

This notebook shows how predictive coding (PC) can be implemented in JAX. Any place you see a **Task** you should complete it. Any place you see ❓ **Questions you may be asked**, you should think about it because it may come up in a discussion!

The aim of this notebook is to give you a flavor of the type of work done in the [ARNI](https://arni-institute.org/) Biological Learning Working Group. The code below is a stripped-down copy of what actually runs in our research codebase, so if you can follow this notebook you can read the real thing.

We will build and train a feedforward neural network on the MNIST dataset using PC (without backpropagation).

JAX is basically NumPy but with automatic differentiation and vectorization built in. To learn more about JAX, you can take a look at their quickstart guide [here](https://docs.jax.dev/en/latest/101/index.html#jax-101), but we will explain some things along the way. We will also use: (i) [Equinox](https://docs.kidger.site/equinox/), which allows you to define neural nets with PyTorch-like syntax; and (ii) [Optax](https://optax.readthedocs.io/en/latest/index.html), which provides optimizers such as gradient descent and Adam.

# Task 1: Setting up the environment
Install any necessary dependencies to run the next cell. You might consider using a virtual environment manager such as [`uv`](https://docs.astral.sh/uv/).

Run the setup cell below to install the JAX dependencies into this notebook's Python environment. The version pins keep them compatible with the runtime's existing TensorFlow and NumPy 1.x dependencies. Restart the kernel after installation, then run the imports.

**❓Questions you may be asked:**
- What is a virtual environment?
- Why is it useful?

In [2]:
%pip install "numpy<2" "ml-dtypes==0.2.0" "jax[cpu]==0.4.23" "equinox==0.11.3" "optax==0.1.9" "chex==0.1.86"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 30.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.2/197.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.2/77.2 MB 15.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 9.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from typing import Callable

import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import optax

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import warnings
warnings.simplefilter("ignore")

## Hyperparameters

We define some hyperparameters related to the data, network, optimizers, etc.

In [ ]:
SEED = 42

INPUT_DIM, OUTPUT_DIM = 28 * 28, 10
HIDDEN_SIZE = 300
N_LAYERS = 3  # number of weight layers

ACTIVITY_LR = 0.1 # settling / inference learning rate
INFERENCE_STEPS = 20 # number of inference steps per batch

PARAM_LR = 1e-3 # parameter learning rate
BATCH_SIZE = 64

TEST_EVERY = 50 # evaluate every 50 batches
N_TRAIN_ITERS = 500

## Dataset

Some utils to fetch MNIST.

In [ ]:
def get_mnist_loaders(batch_size, seed=SEED):
    train_data = MNIST(train=True, normalize=True)
    test_data = MNIST(train=False, normalize=True)
    train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        generator=torch.Generator().manual_seed(seed),
    )
    test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
    )
    return train_loader, test_loader


class MNIST(datasets.MNIST):
    def __init__(self, train, normalize=True, save_dir="data"):
        if normalize:
            transform = transforms.Compose(
                [
                    transforms.ToTensor(),
                    transforms.Normalize(
                        mean=(0.1307), std=(0.3081)
                    )
                ]
            )
        else:
            transform = transforms.Compose([transforms.ToTensor()])
        super().__init__(save_dir, download=True, train=train, transform=transform)

    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        img = torch.flatten(img)
        label = one_hot(label)
        return img, label


def one_hot(labels, n_classes=10):
    arr = torch.eye(n_classes)
    return arr[labels]


## Predictive Coding 101
Predictive coding (PC) is an algorithm for *inference* and *learning* that optimizes a prediction-error energy function [[rao1999](https://www.nature.com/articles/nn0199_79), [whittington2017](https://direct.mit.edu/neco/article/29/5/1229/8261/An-Approximation-of-the-Error-Backpropagation), [pinchetti2025](https://arxiv.org/abs/2407.01163)]. It is an alternative to backpropagation and uses local Hebbian learning rules to update its weights, which only requires pre- and postsynaptic activity, making it a spatially local algorithm [[bogacz2017](https://pmc.ncbi.nlm.nih.gov/articles/PMC5341759/)]. In its most common form, each layer of the network has a latent variable $\boldsymbol{z}$ that is optimized to minimize a global energy function—a process called inference (also referred to algorithmically as settling). Once the latent variables have settled, the weights are updated to minimize the same energy function—learning. A standard PC assumption is that each layer's latent is a Gaussian random variable with a mean given by the output of the previous layer. This allows one to write down a probabilistic model and derive a resulting energy function from the log probability. For example, the probability distribution, and resulting energy function, for an $\ell$-layer feedforward network with latents $(\boldsymbol{z}_0=\boldsymbol{x}), \boldsymbol{z}_1, \ldots, (\boldsymbol{z}_{\ell}=\boldsymbol{y})$, weights $\boldsymbol{W}_1, \ldots, \boldsymbol{W}_{\ell}$, and activation function $g$ are
$$
\begin{gather}
p(\boldsymbol{z}_0, \boldsymbol{z}_1, \ldots, \boldsymbol{z}_{\ell}) = p(\boldsymbol{z}_1\mid \boldsymbol{z}_0) \prod_{i=2}^{\ell} p(\boldsymbol{z}_i \mid \boldsymbol{z}_{i-1}),
\quad
p(\boldsymbol{z}_i \mid \boldsymbol{z}_{i-1}) = \mathcal{N}(\boldsymbol{z}_i; \boldsymbol{W}_i g(\boldsymbol{z}_{i-1}), \mathbf{I}), \\
E(\boldsymbol{z}_0, \boldsymbol{z}_1, \ldots, \boldsymbol{z}_{\ell}) = -\log p(\boldsymbol{z}_0, \boldsymbol{z}_1, \ldots, \boldsymbol{z}_{\ell}) = \frac12\sum_{i=1}^{\ell} \lVert \boldsymbol{z}_i - \boldsymbol{W}_i g(\boldsymbol{z}_{i-1})\rVert^2.
\end{gather}
$$
Note that we have denoted the data input as $\boldsymbol{z}_0=\boldsymbol{x}$ and its label as $\boldsymbol{z}_{\ell}=\boldsymbol{y}$.

We are now going to code this up in JAX.

In [ ]:
class FeedforwardPCModel(eqx.Module):
    # Each layer is Sequential([activation, Linear]): a PRE-activation layer.
    layers: list[eqx.Module]

    def __init__(
            self,
            key,
            *,
            input_size,
            hidden_size,
            output_size,
            activation_fn,
            input_activation_fn=eqx.nn.Identity(),
            n_layers=2,
            use_bias=True,
    ):
        keys = jr.split(key, n_layers)
        widths = [input_size] + [hidden_size] * (n_layers - 1) + [output_size]

        layers = []
        for i in range(n_layers):
            act = input_activation_fn if i == 0 else activation_fn
            act_layer = act if isinstance(
                act, eqx.Module) else _StaticActivation(act)
            linear = eqx.nn.Linear(widths[i],
                                   widths[i + 1],
                                   key=keys[i],
                                   use_bias=use_bias)
            layers.append(eqx.nn.Sequential([act_layer, linear]))

        self.layers = layers

    def __call__(self, x):
        # feedforward pass for a SINGLE example
        for layer in self.layers:
            x = layer(x)
        return x

    def __getitem__(self, i):
        # Implements indexing so PC loops can write
        # err = activities[l] - model[l](activities[l - 1]).
        return self.layers[i]

    def __len__(self):
        return len(self.layers)


class _StaticActivation(eqx.Module):
    # Not important to understand this class for this tutorial.
    # Wraps a bare callable (e.g. jax.nn.tanh) as an eqx.Module whose function
    # lives in static metadata, not as a pytree leaf.
    fn: Callable = eqx.field(static=True)

    def __call__(self, x, *, key=None):
        return self.fn(x)

Let's instantiate it. JAX uses explicit random number generators (see the [docs](https://jax.readthedocs.io/en/latest/random-numbers.html)): the model takes a `key` and splits it once per layer. Printing an Equinox module shows its pytree structure. The only arrays are the `weight`s and `bias`es.

In [ ]:
key = jr.PRNGKey(SEED)

model = FeedforwardPCModel(
    key,
    input_size=INPUT_DIM,
    hidden_size=HIDDEN_SIZE,
    output_size=OUTPUT_DIM,
    activation_fn=jax.nn.tanh,
    n_layers=N_LAYERS,
)
model

The model acts on a single example. To run it on a batch we vectorize it with `jax.vmap`, one of the core JAX transforms (see the [docs](https://jax.readthedocs.io/en/latest/automatic-vectorization.html)). `vmap` adds a leading batch axis to every input and output without us touching the model code. Let's check both on some MNIST data.

In [ ]:
train_loader, test_loader = get_mnist_loaders(BATCH_SIZE)
img_batch, label_batch = next(iter(train_loader))

# we need to turn the torch.Tensor data into numpy arrays for jax
img_batch, label_batch = img_batch.numpy(), label_batch.numpy()
print(f"image batch: {img_batch.shape}, label batch: {label_batch.shape}")

# a single example
print(f"model(x) for one example: {model(img_batch[0]).shape}")

# the whole batch: vmap adds the batch axis for us
print(f"vmap(model)(X) for the batch: {jax.vmap(model)(img_batch).shape}")

## PC energy and optimization

Recall that input $\boldsymbol{z}_0 = \boldsymbol{x}$ is clamped below the first layer and the target $\boldsymbol{z}_{\ell} = \boldsymbol{y}$ is clamped at the output of the last layer. *Inference* will optimize all the latent variables (i.e., the activities) $\boldsymbol{z}_1, \ldots, \boldsymbol{z}_{\ell-1}$ to minimize the energy function. *Learning* will then update the weights $\boldsymbol{W}_1, \ldots, \boldsymbol{W}_{\ell}$ to minimize the same energy function *after* inference has settled the latents.

\begin{equation}
    \text{Inference:} \quad \Delta \boldsymbol{z}_i \propto - \frac{\partial E}{\partial \boldsymbol{z}_i}
\end{equation}

\begin{equation}
    \text{Learning:} \quad \Delta \boldsymbol{W}_i \propto - \frac{\partial E}{\partial \boldsymbol{W}_i}
\end{equation}

The energy function is a sum of squared errors between each layer's latent and the output of the previous layer passed through the weights and activation function (the energy $E$ above). See [Figure 1 here](https://toddmorrill.github.io/machine%20learning/unsupervised-predictive-coding/) for a diagram of the layerwise circuit.

Let's implement `energy_fn`, which takes the model (i.e., the weights and biases), a list of activities, and one input/output pair, and walks the layers accumulating the squared error at each. Note:
* It is written for a **single example**. Batching comes next.
* It returns a tuple `(energy, aux)`. `aux` is a dictionary of per-layer energies that we will use to monitor inference.
* Python indexes from 0 while the equations index from 1, so `activities[0]` is $\boldsymbol{z}_1$, and `activities[-1]` occupies the slot of $\boldsymbol{z}_\ell$. Since $\boldsymbol{z}_\ell = \boldsymbol{y}$ is clamped, `energy_fn` ignores that entry and uses `output` instead; `activities[-1]`, the feedforward prediction of $\boldsymbol{y}$, is what we use at test time.
* The per-layer keys (`energy/layer_<i>`, `energy/output`) are logged for inspection.

In [ ]:
def energy_fn(model, activities, input_, output, record_aux_details=True):
    # Layer-wise PC energy for a SINGLE example.
    # Layer 0 predicts from input_; interior layers predict from the previous
    # activity; the final layer's prediction is compared against output.
    last_idx = len(model) - 1
    energies = []
    aux = {}
    for layer_idx, layer in enumerate(model.layers):
        if layer_idx == 0 and layer_idx == last_idx:
            # A one-layer model predicts the clamped target directly.
            pred = layer(input_)
            err = output - pred
            key = 'energy/output'
        elif layer_idx == 0:
            pred = layer(input_)
            err = activities[layer_idx] - pred
            key = f'energy/layer_{layer_idx}'
        elif layer_idx == last_idx:
            pred = layer(activities[layer_idx - 1])
            err = output - pred
            key = 'energy/output'
        else:
            pred = layer(activities[layer_idx - 1])
            err = activities[layer_idx] - pred
            key = f'energy/layer_{layer_idx}'
        energies.append(0.5 * jnp.sum(err ** 2))
        if record_aux_details:
            aux[key] = energies[-1]
    return sum(energies), aux

**❓Questions you may be asked:**
- Why isn't this backpropagation?

### Batching the energy with `vmap`

To evaluate the energy on a batch we vectorize `energy_fn` with `jax.vmap`. The `in_axes` argument says, for each positional argument, which axis the batch axis is:

| argument | `in_axes` | meaning |
|---|---|---|
| `model` | `None` | shared across the batch, not batched |
| `activities` | `0` | a *list* of arrays; `vmap` maps over pytrees, so every array in the list gets its leading axis batched |
| `input_`, `output` | `0` | the leading axis is the batch |
| `record_aux_details` | `None` | a static Python bool, passed through unchanged |

`batched_energy_fn` therefore returns one energy **per example** (shape `(batch,)`) plus an `aux` dict whose values are also per example. We then reduce over the batch in two different ways:
* **`batched_energy_sum`** is what we differentiate with respect to the **activities**. Each example's activities appear only in its own energy term, so with a *sum* the gradient for example $i$ is exactly $\partial E_i / \partial \mathbf{z}_{\ell,i}$: the prediction error of that example, independent of how many other examples are in the batch.
* **`batched_energy_mean`** is what we differentiate with respect to the **parameters**. The weights are shared by every example, so their gradient sums contributions from the whole batch; dividing by the batch size makes the parameter gradient (and hence the learning rate) invariant to the batch size.

In [ ]:
def batched_energy_fn(model, activities, input_, output, energy_fn_, record_aux_details=True):
    # vmap a single-example energy_fn over the batch dimension
    return jax.vmap(energy_fn_, in_axes=(None, 0, 0, 0, None))(
        model, activities, input_, output, record_aux_details)


def batched_energy_sum(model, activities, input_, output, energy_fn_, record_aux_details=True):
    # sum-reduce per-example energies (preserves per-example activity grads)
    per_ex, aux = batched_energy_fn(model, activities, input_, output, energy_fn_, record_aux_details)
    return jnp.sum(per_ex), jax.tree_util.tree_map(jnp.sum, aux)


def batched_energy_mean(model, activities, input_, output, energy_fn_, record_aux_details=True):
    # mean-reduce per-example energies (batch-size invariant param grads)
    per_ex, aux = batched_energy_fn(model, activities, input_, output, energy_fn_, record_aux_details)
    return jnp.mean(per_ex), jax.tree_util.tree_map(jnp.mean, aux)

### Initializing the activities

The last ingredient is an initial value for the activities. We use a feedforward pass: `activities[l] = model[l](activities[l - 1])`, starting from the input.

❓ **Questions you may be asked**: Can you think of other ways of initializing the activities? What are the trade-offs?

In [ ]:
def init_activities_with_ffwd(model, input_):
    # feedforward init for a SINGLE example:
    # activities[l] = layer_l(activities[l - 1]), starting from input_
    activities = [model[0](input_)]
    for l in range(1, len(model)):
        activities.append(model[l](activities[l - 1]))
    return activities

Let's test it on the MNIST batch from before. For a single example we call the function directly; for the batch we `vmap` it with `in_axes=(None, 0)`: the model is shared, the input is batched. Note how every activity acquires a leading batch axis.

# Task 2: `vmap` sanity check. Initialize the activities for a batch of MNIST images. The code below is not implemented yet. Implement it and check that the shapes of the activities are as expected.

In [ ]:
# a single example
activities_one = init_activities_with_ffwd(model, img_batch[0])
for l, z in enumerate(activities_one):
    print(f"single example, activity z_{l}: {z.shape}")

# the whole batch
activities = jax.vmap(init_activities_with_ffwd, in_axes=(None, 0))(model, img_batch)
for l, z in enumerate(activities):
    assert z.shape == (img_batch.shape[0],) + activities_one[l].shape
    print(f"batch, activity z_{l}: {z.shape}")

Ok so now we have everything to test our PC energy function: model, activities, and some data. We look at the per-example energies first, then at the batch mean broken down per layer.

In [ ]:
per_example_energy, _ = batched_energy_fn(model, activities, img_batch, label_batch, energy_fn)
print(f"per-example energies: shape {per_example_energy.shape}, first five: {per_example_energy[:5]}")

energy, aux = batched_energy_mean(model, activities, img_batch, label_batch, energy_fn)
print(f"\nbatch-mean energy: {energy:.4f}")
for name, value in aux.items():
    print(f"  {name}: {value:.4f}")

It works! Notice that at the feedforward initialization *every hidden-layer energy is exactly zero*: each activity was set to precisely what the layer below predicts. All of the energy sits in the output term, the mismatch between the network's prediction and the one-hot label. Inference will now spread this error back through the hidden activities.

## Energy gradients

How do we minimize the PC energy we defined above (the energy $E$)? We do this in two phases: first with respect to the activities (inference) and then with respect to the weights (learning).

We are going to use autodiff, which JAX embeds by design (see the [docs](https://jax.readthedocs.io/en/latest/notebooks/autodiff_cookbook.html)). If you're familiar with PyTorch, you are probably used to `loss.backward()` for this. JAX, on the other hand, is functional (as opposed to object-oriented): `jax.grad(f, argnums=1)` *returns a new function* that computes the gradient of `f` with respect to its second argument.

**Technical note**: for the parameters we use Equinox's `filter_grad` rather than JAX's native `grad`. The model pytree contains things that are not arrays (the static activation functions), and `filter_grad` differentiates only with respect to the array leaves of its first argument, while `grad` alone would throw an error.

In [ ]:
# "the gradient of the (summed) energy with respect to the 2nd argument (the activities)"
def compute_activity_grad(model, activities, input_, output, record_aux_details=True):
    # per-example activity gradient of the batch-SUMMED energy
    return jax.grad(batched_energy_sum, argnums=1, has_aux=True)(
        model, activities, input_, output, energy_fn, record_aux_details)


# for parameters, the only differences are: we differentiate the batch-MEAN energy...
# ...with respect to the 1st argument (the model), skipping its non-array leaves

def compute_param_grad(model, activities, input_, output, record_aux_details=True):
    # batch-averaged parameter gradient
    return eqx.filter_grad(batched_energy_mean, has_aux=True)(
        model, activities, input_, output, energy_fn, record_aux_details)

Let's test the activity gradient. Besides the shapes, look at the *size* of the gradient at each layer, right at the feedforward initialization.

In [ ]:
dEdzs, aux = compute_activity_grad(model, activities, img_batch, label_batch)
for l, dEdz in enumerate(dEdzs):
    print(f"activity gradient dE/dz_{l}: shape {dEdz.shape}, mean |dE/dz| = {jnp.abs(dEdz).mean():.4f}")

❓ **Challenge Question, not mandatory**: Only one activity has a non-zero gradient at initialization, and it is not the last one. Explain both facts using the energy: which terms does each $\boldsymbol{z}_\ell$ appear in, and which of those are non-zero at the feedforward initialization? What does this imply about how many inference steps are needed for the output error to reach the first layer?

Now the parameter gradient. `filter_grad` returns a pytree with the same structure as the model: gradient arrays for the weights and biases, and nothing for the static parts. The same reasoning as above tells you which layers get a non-zero gradient at initialization, and hence why inference has to run *before* learning.

In [ ]:
param_grads, aux = compute_param_grad(model, activities, img_batch, label_batch)
for l, layer_grad in enumerate(param_grads.layers):
    linear_grad = layer_grad.layers[1]  # Sequential([activation, Linear]) -> the Linear
    print(f"layer {l}: dE/dW {linear_grad.weight.shape}, dE/db {linear_grad.bias.shape}, "
          f"mean |dE/dW| = {jnp.abs(linear_grad.weight).mean():.2e}")

## Updates

Before putting everything together, let's wrap our gradients into update functions. This will also allow us to use JAX's `jit` primitive (here Equinox's `filter_jit`, which again treats the non-array parts of the model as static), which compiles a function the first time it is called so that subsequent calls run much faster (see the [docs](https://jax.readthedocs.io/en/latest/jit-compilation.html) for more details).

These functions take an Optax optimizer and its state in addition to the model, activities, and data. Optax optimizers are pure functions: `optim.update` returns the updates *and a new state*, and `eqx.apply_updates` adds the updates to the pytree being optimized (a list of arrays for the activities, the model for the parameters).

In [ ]:
@eqx.filter_jit
def update_activities(model, activities, x_optim, x_opt_state, input_, output, record_aux_details=True):
    activity_grads, aux = compute_activity_grad(model, activities, input_, output, record_aux_details)
    updates, x_opt_state = x_optim.update(activity_grads, x_opt_state, params=activities)
    activities = eqx.apply_updates(activities, updates)
    return activities, x_opt_state, aux


# note that the only difference with the above function is...
# ...the variable we are updating (parameters vs. activities)
@eqx.filter_jit
def update_params(model, activities, p_optim, p_opt_state, input_, output, record_aux_details=True):
    param_grads, aux = compute_param_grad(model, activities, input_, output, record_aux_details)
    updates, p_opt_state = p_optim.update(param_grads, p_opt_state, params=model)
    model = eqx.apply_updates(model, updates)
    return model, p_opt_state, aux

Let's watch inference happen. We run a handful of activity updates with plain gradient descent and print the per-layer energies after each step (this is what `aux` is for).

In [ ]:
x_optim = optax.sgd(ACTIVITY_LR)
x_opt_state = x_optim.init(activities)

settled = activities
for step in range(5):
    settled, x_opt_state, _ = update_activities(
        model, settled, x_optim, x_opt_state, img_batch, label_batch)
    total, per_layer = batched_energy_mean(model, settled, img_batch, label_batch, energy_fn)
    print(f"step {step + 1}: total energy {total:.4f}  |  "
          + "  ".join(f"{k.split('/')[1]}: {v:.2e}" for k, v in per_layer.items()))

You can see the output error being pushed down into the hidden layers one layer per step, while the total energy decreases: `layer_1` becomes non-zero after the first update, `layer_0` only after the second.

In practice, we do not run this Python loop—we use `jax.lax.scan`, JAX's compiled `for` loop (see the [docs](https://docs.jax.dev/en/latest/_autosummary/jax.lax.scan.html)). `scan` takes a body function `(carry, _) -> (carry, output)`, threads the `carry` (here the activities and the optimizer state) through `T` iterations, and stacks the per-iteration outputs (here the `aux` energies). The result is a single compiled program per batch: initialize the activities, settle for `T` steps, then update the parameters once.

In [ ]:
@eqx.filter_jit
def train_step(model, input_, output, p_optim, p_opt_state, x_optim, T, record_aux_details=True):
    # one batch: feedforward-initialize, settle for T steps, update the parameters once

    # inference: initialize the activities per example (vmap over the batch)...
    activities = jax.vmap(init_activities_with_ffwd, in_axes=(None, 0))(model, input_)
    x_opt_state = x_optim.init(activities)

    # ...then relax them for T steps
    def settle_body(carry, _):
        activities, x_opt_state = carry
        activities, x_opt_state, aux = update_activities(
            model, activities, x_optim, x_opt_state, input_, output, record_aux_details)
        return (activities, x_opt_state), aux

    (activities, _), aux_scan = jax.lax.scan(
        settle_body, (activities, x_opt_state), None, length=T)

    # energy of the settled state, i.e. what the parameter update sees
    energy, _ = batched_energy_mean(model, activities, input_, output, energy_fn, False)

    # learning: one parameter update at the settled activities
    model, p_opt_state, _ = update_params(
        model, activities, p_optim, p_opt_state, input_, output, record_aux_details=False)

    return {
        "model": model,
        "p_opt_state": p_opt_state,
        "activities": activities,
        "energy": energy,
        "aux": aux_scan,  # per-layer energies along the settling trajectory, each of shape (T,)
    }

# Task 3: Plot the per-layer energies along the settling trajectory for 50 settling steps.
Let's run a single training step and look at the energies along the settling trajectory. Since `scan` stacks the per-step `aux`, each entry is a vector of length `T`: the energy of that layer *before* each of the `T` updates.
The code below is not implemented yet.

In [ ]:
settling_steps = 50
x_optim = optax.sgd(ACTIVITY_LR)
p_optim = optax.adam(PARAM_LR)
p_opt_state = p_optim.init(eqx.filter(model, eqx.is_array))

settling_out = train_step(
    model, img_batch, label_batch, p_optim, p_opt_state, x_optim,
    T=settling_steps, record_aux_details=True,
)

fig, ax = plt.subplots(figsize=(8, 5))
for name, energies in settling_out["aux"].items():
    ax.plot(range(settling_steps), energies, label=name.removeprefix("energy/"))
ax.set(
    xlabel="Settling step (before update)",
    ylabel="Energy (sum over batch)",
    title="Per-layer energies over 50 settling steps",
)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

❓**Questions you may be asked**: What do you observe in this trace? Is there anything problematic about the energy for layer 0?

## Putting everything together: training and testing

Now that we have a training step, we need a way to loop over the data and a way to evaluate.

Prediction is simply the feedforward pass: as discussed, `activities[-1]` from the feedforward initialization is the network's prediction of the label, and it is exactly `model(x)`. We `vmap` and `jit` it.

In [ ]:
@eqx.filter_jit
def predict(model, input_):
    return jax.vmap(model)(input_)


def evaluate(model, test_loader):
    # Count examples rather than averaging batch accuracies: the last batch
    # may be smaller, and every test image should have the same weight.
    n_correct, n_examples = 0, 0
    for img_batch, label_batch in test_loader:
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()
        preds = predict(model, img_batch)
        if not bool(jnp.all(jnp.isfinite(preds))):
            raise FloatingPointError("Non-finite predictions; this configuration diverged.")
        correct = jnp.argmax(label_batch, axis=1) == jnp.argmax(preds, axis=1)
        n_correct += int(jax.device_get(jnp.sum(correct)))
        n_examples += len(img_batch)
    if n_examples == 0:
        raise ValueError("The test loader is empty.")
    return 100.0 * n_correct / n_examples


def train(model, activity_lr, inference_steps, param_lr, batch_size,
          test_every, n_train_iters, *, seed=SEED, activity_optimizer="sgd",
          metrics_callback=None, return_history=False):
    if n_train_iters < 1 or test_every < 1 or inference_steps < 1:
        raise ValueError("n_train_iters, test_every, and inference_steps must be positive.")
    if activity_optimizer == "sgd":
        x_optim = optax.sgd(activity_lr)
    elif activity_optimizer == "adam":
        x_optim = optax.adam(activity_lr)
    else:
        raise ValueError("activity_optimizer must be 'sgd' or 'adam'.")
    p_optim = optax.adam(param_lr)
    p_opt_state = p_optim.init(eqx.filter(model, eqx.is_array))

    train_loader, test_loader = get_mnist_loaders(batch_size, seed=seed)
    train_loader = infinite_loader(train_loader)
    history = []
    for train_iter, (img_batch, label_batch) in enumerate(train_loader, start=1):
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()
        out = train_step(model,
                         img_batch,
                         label_batch,
                         p_optim,
                         p_opt_state,
                         x_optim,
                         T=inference_steps,
                         record_aux_details=False)
        model, p_opt_state = out["model"], out["p_opt_state"]

        # Always evaluate after the final parameter update, even when the
        # requested iteration count is not a multiple of test_every.
        if train_iter % test_every == 0 or train_iter == n_train_iters:
            train_energy = float(jax.device_get(out["energy"]))
            if not bool(jnp.isfinite(train_energy)):
                raise FloatingPointError("Non-finite training energy; this configuration diverged.")
            avg_test_acc = evaluate(model, test_loader)
            metrics = {
                "train/iteration": train_iter,
                "train/energy": train_energy,
                "test/accuracy": avg_test_acc,  # percentage, from 0 to 100
            }
            history.append(metrics)
            if metrics_callback is not None:
                metrics_callback(metrics)
            print(
                f"Train iter {train_iter}, train energy={train_energy:.4f}, "
                f"avg test accuracy={avg_test_acc:.2f}%")
        if train_iter >= n_train_iters:
            break

    return (model, history) if return_history else model


def infinite_loader(dataloader):
    while True:
        yielded = False
        for batch in dataloader:
            yielded = True
            yield batch
        if not yielded:
            raise ValueError("The training loader is empty.")


## Run

Let's test our implementation.

In [ ]:
trained_model = train(
    model=model,
    activity_lr=ACTIVITY_LR,
    inference_steps=INFERENCE_STEPS,
    param_lr=PARAM_LR,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS
)

🥳 Great, we see that our model is learning! This model was not tuned, and you can probably improve the performance by tweaking some of the hyperparameters.

# Task 4: Experiments
Vary the following hyperparameters, run the model with each, and plot the test accuracy after 1,000 iterations as a function of the hyperparameter.

1. Vary `INFERENCE_STEPS` (e.g. 1, 5, 20, 50, 100). How does the accuracy after 1,000 iterations depend on how well the activities are settled?
2. Vary `ACTIVITY_LR` (e.g. 0.001, 0.01, 0.1, 0.5, 1.0). How does the accuracy after 1,000 iterations depend on the activity learning rate?
3. Optional: Replace `optax.sgd` for the activities with `optax.adam`. Does it help? Why might this be a less biologically plausible choice?

❓**Questions you may be asked:**
- What number of inference steps and activity learning rate would you recommend to achieve peak performance and manage runtime costs?
- Can you achieve 98% test set accuracy? Feel free to play around—but remember, no backprop.

### Experiment setup

Run the preceding imports and function-definition cells first. The earlier
500-iteration demo is optional. The cells below define two separate sweeps:
change inference steps while holding activity LR at 0.1, then change activity
LR while holding inference steps at 20. Each run gets a fresh model, fresh
optimizer state, the same seeded training-data order, and 1,000 parameter
updates. Evaluation includes all 10,000 MNIST test images.

`TASK4_SEEDS = [SEED]` gives 10 runs. Use several seeds, for example
`[42, 43, 44]`, to estimate variability; the plots then show the mean and
standard deviation across seeds. Continue to Bonus Task 5 below to run these
experiments with WandB logging and plot results retrieved from its API.


In [ ]:
from time import perf_counter

TASK4_BASE_CONFIG = {
    "seed": SEED,
    "input_dim": INPUT_DIM,
    "output_dim": OUTPUT_DIM,
    "hidden_size": HIDDEN_SIZE,
    "n_layers": N_LAYERS,
    "activation": "tanh",
    "activity_lr": ACTIVITY_LR,
    "inference_steps": INFERENCE_STEPS,
    "param_lr": PARAM_LR,
    "batch_size": BATCH_SIZE,
    "test_every": TEST_EVERY,
    "n_train_iters": 1000,
    "activity_optimizer": "sgd",
}
TASK4_SWEEPS = {
    "inference_steps": [1, 5, 20, 50, 100],
    "activity_lr": [0.001, 0.01, 0.1, 0.5, 1.0],
}
TASK4_SEEDS = [SEED]


def task4_configs():
    for parameter, values in TASK4_SWEEPS.items():
        for value in values:
            for seed in TASK4_SEEDS:
                yield {
                    **TASK4_BASE_CONFIG,
                    parameter: value,
                    "seed": seed,
                    "sweep_parameter": parameter,
                }


def run_task4_experiment(config, metrics_callback=None):
    # Never reuse trained_model from the earlier demonstration.
    torch.manual_seed(config["seed"])
    experiment_model = FeedforwardPCModel(
        jr.PRNGKey(config["seed"]),
        input_size=config["input_dim"],
        hidden_size=config["hidden_size"],
        output_size=config["output_dim"],
        activation_fn=jax.nn.tanh,
        n_layers=config["n_layers"],
    )
    started = perf_counter()
    _, history = train(
        model=experiment_model,
        activity_lr=config["activity_lr"],
        inference_steps=config["inference_steps"],
        param_lr=config["param_lr"],
        batch_size=config["batch_size"],
        test_every=config["test_every"],
        n_train_iters=config["n_train_iters"],
        seed=config["seed"],
        activity_optimizer=config["activity_optimizer"],
        metrics_callback=metrics_callback,
        return_history=True,
    )
    if not history or history[-1]["train/iteration"] != config["n_train_iters"]:
        raise ValueError("Training did not complete the requested number of iterations.")
    return {
        "config": dict(config),
        "history": history,
        "final_test_accuracy": history[-1]["test/accuracy"],
        "runtime_seconds": perf_counter() - started,
    }


# Bonus Task 5: Log your experiments on Wandb
We often want to log our experiments more carefully, and Wandb is a handy tool for this. Log your experiments on Wandb, and pull your test accuracy numbers from their API to make your plots. Not necessary—only if you want to show off your results!

### 1. Configure WandB

WandB is already installed in this workspace. If your kernel does not have
it, run `%pip install wandb` in a separate cell and restart the kernel.
Set `WANDB_ENTITY` to your WandB username or team slug, or leave it as `None`
to use your account's default entity. Set a project name below.

The group identifies this set of comparisons. Save the printed
`TASK4_GROUP` value if you want to fetch these results after restarting the
kernel; replace its generated value with that saved string. Keep the same
group while running and fetching results.

Logging follows [WandB's Python SDK workflow](https://docs.wandb.ai/models/ref/python/functions).
`train/iteration` is the chart's x-axis. Accuracy is logged in percent.
`test/accuracy_at_1000` stores the result after exactly 1,000 updates.
`runtime/seconds` includes training, evaluation, and JIT compilation.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import wandb

WANDB_PROJECT = "pc-in-jax-task4"
WANDB_ENTITY = None  # or "your-username-or-team"
WANDB_MODE = "online"  # use "offline" to save locally and sync later
TASK4_GROUP = (
    f"task4-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}-{uuid4().hex[:6]}"
)
print("Experiment group:", TASK4_GROUP)


def run_task4_wandb(config, *, mode=None, group=None):
    run_name = (
        f"{config.get('sweep_parameter', 'baseline')}-"
        f"T{config['inference_steps']}-lr{config['activity_lr']:g}-"
        f"{config['activity_optimizer']}-seed{config['seed']}"
    )
    with wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        group=group or TASK4_GROUP,
        name=run_name,
        job_type="task4",
        tags=["task4", config.get("sweep_parameter", "baseline")],
        config=dict(config),
        mode=mode or WANDB_MODE,
        dir=".",
    ) as run:
        run.define_metric("train/iteration")
        run.define_metric("train/energy", step_metric="train/iteration")
        run.define_metric("test/accuracy", step_metric="train/iteration")
        result = run_task4_experiment(config, metrics_callback=run.log)
        run.summary.update({
            "test/final_accuracy": result["final_test_accuracy"],
            "train/iterations_completed": result["history"][-1]["train/iteration"],
            "runtime/seconds": result["runtime_seconds"],
        })
        if (config["n_train_iters"] == 1000
                and result["history"][-1]["train/iteration"] == 1000):
            run.summary["test/accuracy_at_1000"] = result["final_test_accuracy"]
        result["run_id"] = run.id
        result["entity"] = run.entity
        result["project"] = run.project
        # The context manager finishes and flushes this run before the next.
    return result


### 2. Test the pipeline locally

This optional two-iteration check uses offline logging and a separate group.
It does not require a login and is excluded from the 1,000-iteration plots.
It writes a run under `wandb/` so you can check the pipeline before starting
all 10 experiments. The first run also compiles the JAX functions.


In [ ]:
smoke_config = {
    **TASK4_BASE_CONFIG,
    "sweep_parameter": "smoke",
    "inference_steps": 1,
    "n_train_iters": 2,
    "test_every": 2,
}
smoke_result = run_task4_wandb(
    smoke_config, mode="offline", group=f"{TASK4_GROUP}-smoke",
)


### 3. Log in and run both sweeps

Run this cell when you are ready to train. `wandb.login()` prompts for your
API key if needed; use its prompt or an existing `WANDB_API_KEY` environment
variable. Keep credentials out of notebook source cells.

With the defaults, this runs five inference-step experiments and five
activity-LR experiments, evaluating every 50 training batches. Other
hyperparameters stay at the baseline. Larger inference-step counts take
more work per batch. Non-finite energies or predictions mark a run as failed;
the loop reports that configuration and continues with the remaining runs.

To run without network access, set `WANDB_MODE = "offline"` first. Upload
the resulting runs later with `wandb sync --sync-all` from this directory
after logging in. API retrieval requires uploaded runs.


In [ ]:
if WANDB_MODE == "online":
    wandb.login()

wandb_results = []
failed_experiments = []
for config in task4_configs():
    parameter = config["sweep_parameter"]
    print(f"\nRunning {parameter}={config[parameter]}, seed={config['seed']}")
    try:
        result = run_task4_wandb(config)
    except FloatingPointError as error:
        failed_experiments.append({"config": config, "error": str(error)})
        print(f"Failed: {error}")
        continue
    wandb_results.append(result)
    print(f"Run {result['run_id']}: {result['final_test_accuracy']:.2f}% "
          f"in {result['runtime_seconds']:.1f} seconds")
print(f"\nCompleted {len(wandb_results)} runs; {len(failed_experiments)} failed.")


### 4. Fetch results from WandB and plot

This uses [`wandb.Api().runs()` and run summaries](https://docs.wandb.ai/guides/track/public-api-guide/)
to obtain the accuracy at 1,000 iterations. It filters to completed runs in
this group, so the smoke test, interrupted runs, and runs from other groups
are excluded. If a configuration was rerun in this group, its newest
completed run per seed is used. Failed or missing configurations are reported.

The plot is saved to `task4_wandb_accuracy.png`. With multiple seeds, the
error bars show the standard deviation across seeds. Compare accuracy with
the printed runtimes when choosing inference steps and activity LR.


In [ ]:
import numpy as np


def fetch_task4_results(*, entity=None, project=None, group=None):
    api = wandb.Api()
    entity = entity or WANDB_ENTITY or api.default_entity
    if not entity:
        raise ValueError("Set WANDB_ENTITY to the username or team owning the project.")
    runs = api.runs(
        f"{entity}/{project or WANDB_PROJECT}",
        filters={
            "group": group or TASK4_GROUP,
            "state": "finished",
            "config.n_train_iters": 1000,
        },
        order="-created_at",
    )
    newest_results = {}
    for run in runs:
        config = run.config
        parameter = config.get("sweep_parameter")
        accuracy = run.summary.get("test/accuracy_at_1000")
        if parameter not in TASK4_SWEEPS or accuracy is None:
            continue
        if run.summary.get("train/iterations_completed") != 1000:
            continue
        # Keep each comparison at the configured baseline and seed set.
        if (config.get(parameter) not in TASK4_SWEEPS[parameter]
                or config.get("seed") not in TASK4_SEEDS):
            continue
        if any(config.get(name) != value
               for name, value in TASK4_BASE_CONFIG.items()
               if name not in ("seed", parameter)):
            continue
        try:
            accuracy = float(accuracy)
        except (TypeError, ValueError):
            continue
        if not np.isfinite(accuracy) or not 0.0 <= accuracy <= 100.0:
            continue
        key = (parameter, config[parameter], config["seed"])
        newest_results.setdefault(key, {
            "sweep_parameter": parameter,
            "value": config[parameter],
            "seed": config["seed"],
            "test_accuracy": float(accuracy),
            "runtime_seconds": run.summary.get("runtime/seconds"),
            "run_id": run.id,
        })
    if not newest_results:
        raise ValueError(
            "No completed 1,000-iteration results found. Check the entity, "
            "project, and group, and sync any offline runs first."
        )
    expected = {
        (parameter, value, seed)
        for parameter, values in TASK4_SWEEPS.items()
        for value in values
        for seed in TASK4_SEEDS
    }
    missing = expected - newest_results.keys()
    if missing:
        print("Missing or failed (parameter, value, seed):", sorted(missing))
    return [newest_results[key] for key in sorted(newest_results)]


def plot_task4_results(results):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, parameter in zip(axes, TASK4_SWEEPS):
        by_value = {}
        for result in results:
            if result["sweep_parameter"] == parameter:
                by_value.setdefault(result["value"], []).append(result["test_accuracy"])
        values = sorted(by_value)
        means = [np.mean(by_value[value]) for value in values]
        stds = [np.std(by_value[value], ddof=1) if len(by_value[value]) > 1 else 0.0
                for value in values]
        if values:
            ax.errorbar(values, means, yerr=stds, marker="o", capsize=4)
        else:
            ax.text(0.5, 0.5, "No completed runs", ha="center", transform=ax.transAxes)
        ax.set(xlabel=parameter, ylabel="Test accuracy at 1,000 iterations (%)")
        ax.set_title(
            "Activity LR fixed at " + str(TASK4_BASE_CONFIG["activity_lr"])
            if parameter == "inference_steps" else
            "Inference steps fixed at " + str(TASK4_BASE_CONFIG["inference_steps"])
        )
        if parameter == "activity_lr":
            ax.set_xscale("log")
        ax.set_xticks(TASK4_SWEEPS[parameter])
        ax.set_xticklabels([str(value) for value in TASK4_SWEEPS[parameter]])
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("task4_wandb_accuracy.png", dpi=180, bbox_inches="tight")
    plt.show()
    return fig


task4_api_results = fetch_task4_results()
for row in task4_api_results:
    print(row)
task4_figure = plot_task4_results(task4_api_results)


### Optional: compare activity SGD with Adam

Does Adam help **settling**, **test accuracy**, or **runtime**? These need
separate measurements. Run the earlier imports, hyperparameters, function
definitions, and **Experiment setup** cell (`TASK4_BASE_CONFIG`) first;
the demo, earlier sweeps, and WandB cells are optional.

The defaults compare SGD and Adam at activity learning rates 0.001, 0.01,
and 0.1, using 20 inference steps and 1,000 training updates for each run.
This includes a direct replacement at the baseline LR, plus an equal-size
LR search for each optimizer. Within each seed, initial weights and the
sequence of training batches match. The parameter optimizer stays Adam
at `PARAM_LR`; only the activity optimizer changes. `train_step` initializes
fresh activity optimizer state **for every batch**, retaining Adam's
moments only during that batch's settling steps.

One seed gives six training runs. Set `OPTIMIZER_SEEDS = [42, 43, 44]`
for paired repeats. All runs evaluate the full MNIST test set. By default,
results stay local; optionally enable the existing WandB logger below.


In [ ]:
import json
from pathlib import Path
import numpy as np

OPTIMIZER_BASE_CONFIG = dict(TASK4_BASE_CONFIG)
OPTIMIZER_LRS = sorted({0.001, 0.01, OPTIMIZER_BASE_CONFIG["activity_lr"]})
OPTIMIZER_SEEDS = list(TASK4_SEEDS)
OPTIMIZER_USE_WANDB = False  # If True, first run the "Configure WandB" cell.
OPTIMIZER_RESULTS_PATH = Path("task4_activity_optimizer_results.json")


def activity_optimizer_configs():
    if (not OPTIMIZER_LRS or not OPTIMIZER_SEEDS
            or any(not np.isfinite(lr) or lr <= 0 for lr in OPTIMIZER_LRS)):
        raise ValueError("Choose positive, finite learning rates and at least one seed.")
    if (len(set(OPTIMIZER_LRS)) != len(OPTIMIZER_LRS)
            or len(set(OPTIMIZER_SEEDS)) != len(OPTIMIZER_SEEDS)):
        raise ValueError("Learning rates and seeds must be unique.")
    for seed in OPTIMIZER_SEEDS:
        for lr in OPTIMIZER_LRS:
            for optimizer in ("sgd", "adam"):
                yield {
                    **OPTIMIZER_BASE_CONFIG,
                    "seed": seed,
                    "activity_lr": lr,
                    "activity_optimizer": optimizer,
                    "sweep_parameter": "activity_optimizer",
                }


def run_activity_optimizer_comparison(*, use_wandb=False,
                                      results_path=OPTIMIZER_RESULTS_PATH):
    configs = list(activity_optimizer_configs())
    if use_wandb and WANDB_MODE == "online":
        wandb.login()
    results, failures = [], []
    for index, config in enumerate(configs, start=1):
        print(f"\nRun {index}/{len(configs)}: "
              f"{config['activity_optimizer'].upper()}, "
              f"activity LR={config['activity_lr']:g}, seed={config['seed']}")
        try:
            if use_wandb:
                result = run_task4_wandb(
                    config, group=f"{TASK4_GROUP}-optimizer-comparison")
            else:
                result = run_task4_experiment(config)
        except FloatingPointError as error:
            failures.append({"config": config, "error": str(error)})
            print(f"Failed: {error}")
        else:
            results.append(result)
            print(f"Final accuracy: {result['final_test_accuracy']:.2f}%; "
                  f"runtime: {result['runtime_seconds']:.1f} s")
        # Save after every run so completed measurements survive interruption.
        Path(results_path).write_text(json.dumps({
            "planned_configs": configs, "results": results, "failures": failures,
        }, indent=2))
    print(f"\nCompleted {len(results)} runs; {len(failures)} failed. "
          f"Saved to {results_path}.")
    return results, failures


#### 1. Compare settling on a fixed model and batch

This quick diagnostic holds the weights fixed and starts every optimizer
from the same feedforward activities on the same training batch. It records
mean energy at step 0 and **after** each activity update, including the last
one. Compare the energy after the chosen inference budget and look for
overshooting. This is one untrained model and batch, so the training
experiment below is still needed to assess generalization.


In [ ]:
@eqx.filter_jit
def activity_settling_trace(model, inputs, targets, *, optimizer, lr, steps):
    if steps < 1:
        raise ValueError("steps must be positive.")
    if optimizer == "sgd":
        x_optim = optax.sgd(lr)
    elif optimizer == "adam":
        x_optim = optax.adam(lr)
    else:
        raise ValueError("optimizer must be 'sgd' or 'adam'.")
    activities = jax.vmap(init_activities_with_ffwd, in_axes=(None, 0))(model, inputs)
    x_opt_state = x_optim.init(activities)
    initial_energy, _ = batched_energy_mean(
        model, activities, inputs, targets, energy_fn, False)

    def settle(carry, _):
        activities, state = carry
        activities, state, _ = update_activities(
            model, activities, x_optim, state, inputs, targets, False)
        energy, _ = batched_energy_mean(
            model, activities, inputs, targets, energy_fn, False)
        return (activities, state), energy

    _, energies = jax.lax.scan(settle, (activities, x_opt_state), None, length=steps)
    return jnp.concatenate((jnp.atleast_1d(initial_energy), energies))


diagnostic_config = OPTIMIZER_BASE_CONFIG
diagnostic_seed = OPTIMIZER_SEEDS[0]
diagnostic_model = FeedforwardPCModel(
    jr.PRNGKey(diagnostic_seed),
    input_size=diagnostic_config["input_dim"],
    hidden_size=diagnostic_config["hidden_size"],
    output_size=diagnostic_config["output_dim"],
    activation_fn=jax.nn.tanh,
    n_layers=diagnostic_config["n_layers"],
)
diagnostic_loader, _ = get_mnist_loaders(
    diagnostic_config["batch_size"], seed=diagnostic_seed)
diagnostic_inputs, diagnostic_targets = (
    batch.numpy() for batch in next(iter(diagnostic_loader)))
optimizer_energy_traces = {}
fig, ax = plt.subplots(figsize=(9, 5))
for optimizer, style in (("sgd", "-"), ("adam", "--")):
    for lr in OPTIMIZER_LRS:
        trace = np.asarray(activity_settling_trace(
            diagnostic_model, diagnostic_inputs, diagnostic_targets,
            optimizer=optimizer, lr=lr, steps=diagnostic_config["inference_steps"]))
        optimizer_energy_traces[(optimizer, lr)] = trace
        ax.plot(np.arange(len(trace)), trace, style,
                label=f"{optimizer.upper()}, LR={lr:g}")
        print(f"{optimizer.upper():4s}, LR={lr:g}: "
              f"initial energy={trace[0]:.5f}, final energy={trace[-1]:.5f}")
ax.set(xlabel="Activity updates completed", ylabel="Mean PC energy per example",
       title="Settling with fixed weights and the same training batch", yscale="log")
ax.legend(ncol=2)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("task4_activity_optimizer_settling.png", dpi=180, bbox_inches="tight")
plt.show()


#### 2. Run the matched training experiments

This cell starts all six runs (with the defaults); it can take several
minutes. Histories, final accuracies, runtimes, and failed configurations
are saved after each run to `task4_activity_optimizer_results.json`.
Running it again starts fresh experiments and replaces that results file.
`OPTIMIZER_USE_WANDB = True` additionally logs to a separate comparison group.


In [ ]:
optimizer_results, optimizer_failures = run_activity_optimizer_comparison(
    use_wandb=OPTIMIZER_USE_WANDB)


#### 3. Summarize accuracy, runtime, and learning curves

The table and plots use the final evaluation after the requested number of
updates. Error bars show the sample standard deviation across seeds (zero
for a single run). The printed Adam-minus-SGD differences pair the same
seeds. A setting must finish **all configured seeds** to enter the best-LR
comparison; missing or failed settings remain visible in the table.

Runtime includes training, evaluation, and JIT compilation, so it measures
the cost of the whole experiment. The LR with the highest mean test accuracy
is described as *best tested*: this is an exploratory comparison using the
test set, not an independent final estimate after validation-set tuning.

To plot saved results after restarting, run the prerequisite definitions and
comparison setup, then load
`optimizer_results = json.loads(OPTIMIZER_RESULTS_PATH.read_text())["results"]`
before running the next cell. Keep the comparison settings the same.


In [ ]:
def summarize_activity_optimizers(results):
    configs = list(activity_optimizer_configs())
    expected = {(c["activity_optimizer"], c["activity_lr"], c["seed"]): c
                for c in configs}
    by_setting = {}
    for result in results:
        config = result["config"]
        key = (config["activity_optimizer"], config["activity_lr"], config["seed"])
        if key not in expected or config != expected[key]:
            raise ValueError("Results do not match the current comparison settings.")
        if key in by_setting:
            raise ValueError(f"Duplicate result for {key}.")
        if (not result["history"]
                or result["history"][-1]["train/iteration"] != config["n_train_iters"]):
            raise ValueError(f"Incomplete training result for {key}.")
        if not np.isfinite(result["final_test_accuracy"]):
            raise ValueError(f"Non-finite accuracy for {key}.")
        by_setting[key] = result

    def mean_std(values):
        return float(np.mean(values)), float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    rows, best = [], {}
    print("Optimizer    LR       Seeds     Test accuracy (%)      Runtime (s)")
    for optimizer in ("sgd", "adam"):
        for lr in OPTIMIZER_LRS:
            runs = [by_setting[(optimizer, lr, seed)] for seed in OPTIMIZER_SEEDS
                    if (optimizer, lr, seed) in by_setting]
            if not runs:
                print(f"{optimizer.upper():9s} {lr:<8g} 0/{len(OPTIMIZER_SEEDS)}       missing/failed")
                continue
            accuracy, accuracy_std = mean_std([r["final_test_accuracy"] for r in runs])
            runtime, runtime_std = mean_std([r["runtime_seconds"] for r in runs])
            row = dict(optimizer=optimizer, lr=lr, n=len(runs), runs=runs,
                       accuracy=accuracy, accuracy_std=accuracy_std,
                       runtime=runtime, runtime_std=runtime_std)
            rows.append(row)
            print(f"{optimizer.upper():9s} {lr:<8g} {len(runs)}/{len(OPTIMIZER_SEEDS)}       "
                  f"{accuracy:6.2f} +/- {accuracy_std:.2f}       "
                  f"{runtime:.1f} +/- {runtime_std:.1f}")
            if (len(runs) == len(OPTIMIZER_SEEDS)
                    and (optimizer not in best or accuracy > best[optimizer]["accuracy"])):
                best[optimizer] = row

    print("\nPaired Adam - SGD accuracy differences (percentage points):")
    for lr in OPTIMIZER_LRS:
        paired_seeds = [seed for seed in OPTIMIZER_SEEDS
                        if all((opt, lr, seed) in by_setting for opt in ("sgd", "adam"))]
        if len(paired_seeds) != len(OPTIMIZER_SEEDS):
            print(f"LR={lr:g}: incomplete pairs; no comparison.")
            continue
        differences = [by_setting[("adam", lr, seed)]["final_test_accuracy"]
                       - by_setting[("sgd", lr, seed)]["final_test_accuracy"]
                       for seed in paired_seeds]
        delta, delta_std = mean_std(differences)
        baseline = " (direct baseline replacement)" if lr == OPTIMIZER_BASE_CONFIG["activity_lr"] else ""
        print(f"LR={lr:g}{baseline}: {delta:+.2f} +/- {delta_std:.2f} pp")

    if len(best) == 2:
        sgd, adam = best["sgd"], best["adam"]
        delta = adam["accuracy"] - sgd["accuracy"]
        print(f"\nBest tested SGD: LR={sgd['lr']:g}, {sgd['accuracy']:.2f}%; "
              f"Adam: LR={adam['lr']:g}, {adam['accuracy']:.2f}%.")
        verdict = "higher" if delta > 0 else "lower" if delta < 0 else "equal"
        print(f"Adam's best tested mean accuracy is {verdict} "
              f"(Adam - SGD: {delta:+.2f} pp).")
    else:
        print("\nNot enough complete settings to compare the best learning rates.")
    if len(OPTIMIZER_SEEDS) == 1:
        print("One seed only: repeat with more paired seeds before claiming a reliable improvement.")
    return rows, best


def plot_activity_optimizer_results(rows, best):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    updates = OPTIMIZER_BASE_CONFIG["n_train_iters"]
    for optimizer, color in (("sgd", "tab:blue"), ("adam", "tab:orange")):
        group = sorted((row for row in rows if row["optimizer"] == optimizer),
                       key=lambda row: row["lr"])
        if group:
            for ax, metric in zip(axes[0], ("accuracy", "runtime")):
                ax.errorbar([row["lr"] for row in group],
                            [row[metric] for row in group],
                            yerr=[row[f"{metric}_std"] for row in group],
                            marker="o", capsize=4, color=color, label=optimizer.upper())
        if optimizer in best:
            chosen = best[optimizer]
            histories = [run["history"] for run in chosen["runs"]]
            iterations = [point["train/iteration"] for point in histories[0]]
            for ax, metric in zip(axes[1], ("test/accuracy", "train/energy")):
                values = np.array([[point[metric] for point in history] for history in histories])
                mean = values.mean(axis=0)
                std = values.std(axis=0, ddof=1) if len(histories) > 1 else np.zeros_like(mean)
                ax.plot(iterations, mean, color=color,
                        label=f"{optimizer.upper()}, LR={chosen['lr']:g}")
                ax.fill_between(iterations, mean - std, mean + std, color=color, alpha=0.15)
    axes[0, 0].set(xlabel="Activity learning rate", xscale="log",
                   ylabel=f"Test accuracy after {updates:,} updates (%)",
                   title="Learning-rate sensitivity")
    axes[0, 1].set(xlabel="Activity learning rate", xscale="log",
                   ylabel="Runtime including evaluation and compilation (s)",
                   title="Cost of each training run")
    axes[1, 0].set(xlabel="Training updates", ylabel="Test accuracy (%)",
                   title="Learning curves at each optimizer's best tested LR")
    axes[1, 1].set(xlabel="Training updates", ylabel="Mean settled energy per example",
                   title="Training energy on each evaluated training batch")
    for ax in axes.flat:
        ax.grid(alpha=0.3)
        if ax.lines:
            ax.legend()
    fig.tight_layout()
    fig.savefig("task4_activity_optimizer_comparison.png", dpi=180, bbox_inches="tight")
    plt.show()
    return fig


optimizer_summary, optimizer_best = summarize_activity_optimizers(optimizer_results)
optimizer_comparison_figure = plot_activity_optimizer_results(optimizer_summary, optimizer_best)


**How to answer “Does it help?”** Start with the printed paired difference
at LR=0.1 for the direct replacement, then compare the best tested LRs and
runtimes. Positive accuracy differences favor Adam. Use the fixed-weight
trace to discuss settling: lower energy within the same number of updates
means better minimization on that batch. Energy during training is measured
on different learned models, and lower energy alone does not establish
better test accuracy. Repeat across seeds and, if needed, extend the LR grid
equally for both optimizers before drawing a broad conclusion.

**Why might Adam be less biologically plausible?** Plain activity SGD uses
the current energy gradient with a fixed step size. Adam additionally keeps
a moving average of each activity's gradient and squared gradient, corrects
their initialization bias using a step counter, and normalizes updates by
a square root. Those extra memory and arithmetic requirements make the
proposed neural mechanism more elaborate. They do not turn this into
backpropagation through the settling trajectory: the same PC activity
gradients and fixed-activity parameter update are still used.


The original scaffold of this notebook comes from [JPC](https://github.com/thebuckleylab/jpc) and we thank them for a terrific starting point.